# LightFM Model

### Unit 1 - 환경 및 공통 설정

In [1]:
# Unit 1
from pathlib import Path
import random

import numpy as np
import pandas as pd

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k

SEED = 42
TEST_RATIO = 0.2
EPOCHS = 30
NUM_THREADS = 2

random.seed(SEED)
np.random.seed(SEED)

# DATA_PATH = Path("review_by_llm.csv")
print({"seed": SEED, "test_ratio": TEST_RATIO, "epochs": EPOCHS})

c:\dev\project\SKN27-FINAL-1Team\ai\experiments\.venv\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


{'seed': 42, 'test_ratio': 0.2, 'epochs': 30}


### Unit 2 - 데이터 로드 및 필수 컬럼 검증

In [2]:
# 데이터 프레임 로드
review_df = pd.read_csv("review_by_llm.csv")
recipe_df = pd.read_csv("recipe_fix.csv")
ingredient_alias_df = pd.read_csv("recipe_ingredient_alias.csv")

recipe_cols = [
    "RCP_SNO", "CKG_NM", "INQ_CNT", "SRAP_CNT", "CKG_MTH_ACTO_NM",
    "CKG_STA_ACTO_NM", "CKG_MTRL_ACTO_NM", "CKG_KND_ACTO_NM",
    "CKG_INBUN_NM", "CKG_DODF_NM", "CKG_TIME_NM",
]
ingredient_alias_cols = [
    "RCP_SNO", "CKG_NM", "ingredients_raw", "aliases_matched",
    "ingredients_normalized", "others_count", "others_items",
    "basic_count", "basic_items",
]
review_cols = [
    "recipe_id", "group_id", "star_count", "content",
    "positive", "negative", "star_norm",
]

for name, frame, cols in [
    ("recipe_fix.csv", recipe_df, recipe_cols),
    ("recipe_ingredient_alias.csv", ingredient_alias_df, ingredient_alias_cols),
    ("review_by_llm.csv", review_df, review_cols),
]:
    missing_cols = [c for c in cols if c not in frame.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {name}: {missing_cols}")

recipe_df = recipe_df[recipe_cols].copy()
ingredient_alias_df = ingredient_alias_df[ingredient_alias_cols].copy()
review_df = review_df[review_cols].copy()

print("recipe_cols:", recipe_df.columns.tolist())
print("ingredient_alias_cols:", ingredient_alias_df.columns.tolist())
print("review_cols:", review_df.columns.tolist())

recipe_cols: ['RCP_SNO', 'CKG_NM', 'INQ_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM', 'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM', 'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM']
ingredient_alias_cols: ['RCP_SNO', 'CKG_NM', 'ingredients_raw', 'aliases_matched', 'ingredients_normalized', 'others_count', 'others_items', 'basic_count', 'basic_items']
review_cols: ['recipe_id', 'group_id', 'star_count', 'content', 'positive', 'negative', 'star_norm']


### Unit 3 - interaction 타겟 생성
- 사용자 데이터에 필요한 값들 가공 처리
- 레시피 데이터에 필요한 값들 가공 처리

> 유저 데이터 처리

In [3]:
# 유저 데이터 전처리 - 별점 데이터 정리
# 강의 내용과 다르게 지금 수집한 내용 기준으로 정리, 필요한 경우 단계 나눠서 실험하면서 진행
# 값은 변환값 쓰지 않고 직접 계산해서 사용 
# star_count : 1 ~ 5점 데이터를 -1 ~ +1 사이 값으로 변환 ( -3 이후 * 1/2)
# review_df에서 별점 관련 데이터 드랍 및 필요한 칼럼만 남김

# 별점 데이터 관련 칼럼
star_related_cols = ["star_count", "star_norm"]

# 별점 값 가공해서 신규 컬럼으로 추가 (나중에 해당 부분에서 계산할 수 있도록)
# star_count 값을 -1 ~ +1 구간으로 변환해서 star 컬럼에 추가
# 변환식: ((star_count - 3) / 2)
review_df["star"] = review_df["star_count"].astype(float).apply(lambda x: (x - 3) / 2)


# review_df에서 별점 관련 칼럼 드랍
review_df = review_df.drop(columns=star_related_cols, errors="ignore")

# 필요한 값만 가공해서 사용: 여기서는 별점 관련 데이터가 사라지고
# 남은 칼럼만 유지됨을 확인
print("가공/드랍 이후 review_df 칼럼:", review_df.columns.tolist())

가공/드랍 이후 review_df 칼럼: ['recipe_id', 'group_id', 'content', 'positive', 'negative', 'star']


In [4]:
# 유저 데이터 전처리 - 감성 데이터 정리 
# 긍정 - 부정 값을 감성분석 값으로 사용 (긍정인 경우 +, 부정인 경우 - 나오도록 처리)
# 컬럼은 sentiment 컬럼으로 추가하고 기존 감성분석 컬럼은 제거 

# 감성 데이터 전처리
# 'positive', 'negative' 값을 활용하여 sentiment 컬럼 생성
# sentiment = positive - negative (긍정은 +, 부정은 -)
review_df["sentiment"] = review_df["positive"].astype(float) - review_df["negative"].astype(float)

# 기존 감성 관련 컬럼(drop)
sentiment_related_cols = ["positive", "negative", "neutral", "compound"]
review_df = review_df.drop(columns=sentiment_related_cols, errors="ignore")

# 결과 칼럼 확인
print("감성 처리/드랍 이후 review_df 칼럼:", review_df.columns.tolist())

감성 처리/드랍 이후 review_df 칼럼: ['recipe_id', 'group_id', 'content', 'star', 'sentiment']


In [5]:
# 리뷰 내용은 일단 제거 (content 컬럼)
# 리뷰 내용 제거 (content 컬럼 드랍)
review_df = review_df.drop(columns=["content"], errors="ignore")
print("content 컬럼 제거 후 review_df 칼럼:", review_df.columns.tolist())

content 컬럼 제거 후 review_df 칼럼: ['recipe_id', 'group_id', 'star', 'sentiment']


> 레시피 데이터 처리

In [6]:
# recipe_df, ingredient_alias_df를 RCP_SNO 기준으로 병합
# 중복 제거의 의미는 "중복 행"이 아니라 "겹치는 컬럼명" 처리(CKG_NM 등)

# recipe_df에 이미 있는 공통 컬럼은 ingredient_alias_df에서 제외하고 머지
recipe_base_cols = set(recipe_df.columns)
alias_merge_cols = [
    col for col in ingredient_alias_df.columns
    if col == "RCP_SNO" or col not in recipe_base_cols
]

ingredient_alias_for_merge = ingredient_alias_df[alias_merge_cols].copy()

recipe_df = recipe_df.merge(ingredient_alias_for_merge, on="RCP_SNO", how="left")

print("재료머지 이후 recipe_df 칼럼:", recipe_df.columns.tolist())
print(recipe_df.head())

재료머지 이후 recipe_df 칼럼: ['RCP_SNO', 'CKG_NM', 'INQ_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM', 'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM', 'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM', 'ingredients_raw', 'aliases_matched', 'ingredients_normalized', 'others_count', 'others_items', 'basic_count', 'basic_items']
   RCP_SNO   CKG_NM  INQ_CNT  SRAP_CNT CKG_MTH_ACTO_NM CKG_STA_ACTO_NM  \
0  7016814     된장수육     1396         1              삶기             술안주   
1  7016815  우거지 감자탕     4008        29             끓이기              해장   
2  7016816     만두전골     6350         6             끓이기            손님접대   
3  7016817    무수분보쌈     1829         6              삶기            손님접대   
4  7016820   참치 카나페     4259        10              기타              일상   

  CKG_MTRL_ACTO_NM CKG_KND_ACTO_NM CKG_INBUN_NM CKG_DODF_NM CKG_TIME_NM  \
0             돼지고기            메인반찬          2인분          고급       2시간이내   
1             돼지고기             국/탕          4인분          중급       2시간이내   
2            가

In [7]:
recipe_df.columns

Index(['RCP_SNO', 'CKG_NM', 'INQ_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM',
       'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM',
       'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM', 'ingredients_raw',
       'aliases_matched', 'ingredients_normalized', 'others_count',
       'others_items', 'basic_count', 'basic_items'],
      dtype='object')

In [8]:
# 레시피 컬럼 개별 정리 
# 사용할 컬럼의 경우 컬럼 이름 수정 
# RCP_SNO -> recipe_id
# CKG_NM -> recipe_name
# INQ_CNT -> view_count
# SRAP_CNT -> scrap_count
# CKG_MTH_ACTO_NM -> cooking_method
# CKG_STA_ACTO_NM -> cooking_category
# CKG_MTRL_ACTO_NM -> main_ingred
# CKG_INBUN_NM -> dishes
# CKG_DODF_NM -> cooking_level
# CKG_TIME_NM -> cooking_time
# ingredients_raw -> 컬럼 드랍 (정규화 이름만 사용)
# aliases_matched -> aliases
# ingredients_normalized -> ingredients
# others_items -> 컬럼 드랍 (정규화된 재료 기준으로 유사도 비교)
# basic_items -> 컬럼 드랍 (기본 재료가 몇개 있는지만 체크)
# 위 기준으로 데이터 정리

# 컬럼명 매핑 정의
column_rename_map = {
    "RCP_SNO": "recipe_id",
    "CKG_NM": "recipe_name",
    "INQ_CNT": "view_count",
    "SRAP_CNT": "scrap_count",
    "CKG_MTH_ACTO_NM": "cooking_method",
    "CKG_STA_ACTO_NM": "cooking_category",
    "CKG_MTRL_ACTO_NM": "main_ingred",
    "CKG_INBUN_NM": "dishes",
    "CKG_DODF_NM": "cooking_level",
    "CKG_TIME_NM": "cooking_time",
    "aliases_matched": "aliases",
    "ingredients_normalized": "ingredients"
}

# 드랍할 컬럼 정의
columns_to_drop = [
    "ingredients_raw",       # 원본 재료명
    "others_items",          # 기타 재료 리스트
    "basic_items"            # 기본 재료 리스트
]

# 컬럼명 변경
recipe_df = recipe_df.rename(columns=column_rename_map)

# 필요없는 컬럼 드랍
recipe_df = recipe_df.drop(columns=[col for col in columns_to_drop if col in recipe_df.columns])

# 결과 컬럼 확인
print("정리 이후 recipe_df 컬럼:", recipe_df.columns.tolist())

정리 이후 recipe_df 컬럼: ['recipe_id', 'recipe_name', 'view_count', 'scrap_count', 'cooking_method', 'cooking_category', 'main_ingred', 'CKG_KND_ACTO_NM', 'dishes', 'cooking_level', 'cooking_time', 'aliases', 'ingredients', 'others_count', 'basic_count']


In [9]:
# Hybrid (아이템 피처 포함) 모드에서 alias_id, ingredients 정규화 처리

import ast

def str_to_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

def normalize_aliases(values):
    raw_list = str_to_list(values)
    normalized = []
    for item in raw_list:
        if isinstance(item, dict):
            alias_id = item.get("alias_id")
            alias_name = item.get("name")
            token = alias_id if alias_id else alias_name
            if token:
                normalized.append(str(token))
        elif isinstance(item, str):
            normalized.append(item)
    return sorted(set(normalized))

def normalize_ingredients(values):
    raw_list = str_to_list(values)
    normalized = []
    for item in raw_list:
        # ingredients_normalized 형태: ["재료명", "수량", "단위"]
        if isinstance(item, list) and len(item) > 0:
            name = item[0]
            if name:
                normalized.append(str(name))
        elif isinstance(item, str):
            normalized.append(item)
    return sorted(set(normalized))

recipe_df["aliases"] = recipe_df["aliases"].apply(normalize_aliases)
recipe_df["ingredients"] = recipe_df["ingredients"].apply(normalize_ingredients)

print(recipe_df[["recipe_id", "aliases", "ingredients"]].head())

   recipe_id                                            aliases  \
0    7016814  [alias_0157, alias_0251, alias_0559, alias_061...   
1    7016815  [alias_0036, alias_0099, alias_0140, alias_015...   
2    7016816  [alias_0038, alias_0069, alias_0140, alias_023...   
3    7016817  [alias_0134, alias_0156, alias_0164, alias_016...   
4    7016820  [alias_0103, alias_0165, alias_0233, alias_036...   

                                         ingredients  
0                           [된장, 무, 삼겹살, 술, 콩나물, 홍어]  
1  [간장, 감자, 고추장, 고춧가루, 깻잎, 다시다, 대파, 돼지등뼈, 들깨가루, 마...  
2  [간장, 느타리버섯, 다시마, 대파, 마늘, 만두, 물, 양파, 참치액, 청경채, ...  
3  [꿀, 대파, 레몬즙, 마늘, 맛술, 사과, 삼겹살, 생강가루, 설탕, 양파, 올리...  
4            [귤, 마요네즈, 방울토마토, 설탕, 참치, 체다치즈, 크래커, 후추]  


### Unit 4 - ID 매핑 및 Dataset 구성
- 내부 인덱스 매핑을 고정한다. (`group_id`, `recipe_id`)

In [10]:
# 데이터 매트릭스로 정리하기 전 검사 
# 키 정합성: group_id, recipe_id null/빈값 제거, 문자열 캐스팅
# 중복 pair 처리: 같은 (group_id, recipe_id)가 여러 개면 집계 규칙 1개로 고정(평균/최대/최근)
# recipe_df와 review_df 각각 id 검사 및 매핑이 정상적으로 연결되는지 확인

# 1. Null/빈값 검사
missing_recipe_ids = recipe_df["recipe_id"].isnull().sum() + (recipe_df["recipe_id"].astype(str).str.strip() == "").sum()
missing_review_recipe_ids = review_df["recipe_id"].isnull().sum() + (review_df["recipe_id"].astype(str).str.strip() == "").sum()
missing_review_group_ids = review_df["group_id"].isnull().sum() + (review_df["group_id"].astype(str).str.strip() == "").sum()

print(f"[검사] recipe_df 'recipe_id' null 또는 빈값: {missing_recipe_ids}")
print(f"[검사] review_df 'recipe_id' null 또는 빈값: {missing_review_recipe_ids}")
print(f"[검사] review_df 'group_id' null 또는 빈값: {missing_review_group_ids}")

# 2. 연결 ID 정합성 확인
recipe_id_set = set(recipe_df["recipe_id"].astype(str).str.strip())
review_recipe_id_set = set(review_df["recipe_id"].astype(str).str.strip())
review_group_id_set = set(review_df["group_id"].astype(str).str.strip())

# review_df의 recipe_id가 recipe_df에 존재
unmatched_recipe_ids = review_recipe_id_set - recipe_id_set
print(f"[검사] review_df에서 recipe_df에 없는 recipe_id 개수: {len(unmatched_recipe_ids)}")
if len(unmatched_recipe_ids) > 0:
    print(f"(샘플) 미존재 recipe_id: {list(unmatched_recipe_ids)[:5]}")

# review_df의 group_id 및 recipe_id 조합 고유성 검사 (필요 시)
duplicated_pairs = review_df.duplicated(subset=["group_id", "recipe_id"], keep=False).sum()
print(f"[검사] review_df에서 (group_id, recipe_id) 중복 건수: {duplicated_pairs}")

# 정상적으로 연결되는지 확인 결과 샘플
matched_count = review_df[review_df["recipe_id"].astype(str).str.strip().isin(recipe_id_set)].shape[0]
print(f"[연결 확인] review_df에서 recipe_df로 매핑되는 review 개수: {matched_count} / {review_df.shape[0]}")

[검사] recipe_df 'recipe_id' null 또는 빈값: 0
[검사] review_df 'recipe_id' null 또는 빈값: 0
[검사] review_df 'group_id' null 또는 빈값: 0
[검사] review_df에서 recipe_df에 없는 recipe_id 개수: 14
(샘플) 미존재 recipe_id: ['7034774', '7024219', '7024803', '7034735', '7026841']
[검사] review_df에서 (group_id, recipe_id) 중복 건수: 0
[연결 확인] review_df에서 recipe_df로 매핑되는 review 개수: 990 / 1007


In [11]:
# 위 검사에서 어느 한쪽이라도 매칭되지 않는 경우 해당 row 드랍
# recipe_df, review_df에서 recipe_id 및 group_id 매칭되지 않는 row 드랍
recipe_id_set = set(recipe_df["recipe_id"].astype(str).str.strip())
review_df = review_df[
    review_df["recipe_id"].astype(str).str.strip().isin(recipe_id_set)
]

# review_df의 group_id에서 빈값, null, str.strip()이 빈 칸인 row도 드랍
review_df = review_df[
    review_df["group_id"].notnull() & (review_df["group_id"].astype(str).str.strip() != "")
]

# recipe_df의 recipe_id도 빈값, null, str.strip()이 빈 칸인 row 드랍
recipe_df = recipe_df[
    recipe_df["recipe_id"].notnull() & (recipe_df["recipe_id"].astype(str).str.strip() != "")
]

In [12]:
# 검사 후 매칭으로 누락되는 데이터 없는지 다시 확인

print(f"[최종] recipe_df shape: {recipe_df.shape}")
print(f"[최종] review_df shape: {review_df.shape}")

# recipe_id 기준으로 review_df에서 recipe_df에 없는 값이 남아있는지 확인
final_recipe_id_set = set(recipe_df["recipe_id"].astype(str).str.strip())
final_review_recipe_id_set = set(review_df["recipe_id"].astype(str).str.strip())
leftover_unmatched_recipe_ids = final_review_recipe_id_set - final_recipe_id_set
print(f"[최종검사] review_df에서 여전히 recipe_df에 없는 recipe_id 개수: {len(leftover_unmatched_recipe_ids)}")
if len(leftover_unmatched_recipe_ids) > 0:
    print(f"예시: {list(leftover_unmatched_recipe_ids)[:5]}")

# review_df에서 빈 group_id, 빈 recipe_id row 수 재확인
missing_group_id = review_df["group_id"].isnull().sum() + (review_df["group_id"].astype(str).str.strip() == "").sum()
missing_review_recipe_id = review_df["recipe_id"].isnull().sum() + (review_df["recipe_id"].astype(str).str.strip() == "").sum()
missing_recipe_id = recipe_df["recipe_id"].isnull().sum() + (recipe_df["recipe_id"].astype(str).str.strip() == "").sum()

print(f"[최종검사] review_df에서 group_id가 없는 row 수: {missing_group_id}")
print(f"[최종검사] review_df에서 recipe_id가 없는 row 수: {missing_review_recipe_id}")
print(f"[최종검사] recipe_df에서 recipe_id가 없는 row 수: {missing_recipe_id}")

[최종] recipe_df shape: (3171, 15)
[최종] review_df shape: (990, 4)
[최종검사] review_df에서 여전히 recipe_df에 없는 recipe_id 개수: 0
[최종검사] review_df에서 group_id가 없는 row 수: 0
[최종검사] review_df에서 recipe_id가 없는 row 수: 0
[최종검사] recipe_df에서 recipe_id가 없는 row 수: 0


In [13]:
# Unit 4 - 정리한 데이터들을 데이터 셋으로 묶음 
user_ids = review_df["group_id"].astype(str).unique().tolist()
item_ids = review_df["recipe_id"].astype(str).unique().tolist()

dataset = Dataset()
dataset.fit(users=user_ids, items=item_ids)

print({"users": len(user_ids), "items": len(item_ids)})

{'users': 821, 'items': 563}


### Unit 5 - interactions matrix 생성
- 학습/평가 입력 sparse matrix를 만든다.

In [14]:
recipe_df.columns

Index(['recipe_id', 'recipe_name', 'view_count', 'scrap_count',
       'cooking_method', 'cooking_category', 'main_ingred', 'CKG_KND_ACTO_NM',
       'dishes', 'cooking_level', 'cooking_time', 'aliases', 'ingredients',
       'others_count', 'basic_count'],
      dtype='object')

In [15]:
review_df.columns

Index(['recipe_id', 'group_id', 'star', 'sentiment'], dtype='object')

In [16]:
# 학습용 매트릭스를 만드는 부분
# interaction_value는 star + sentiment 더해서 만든다. (나중에 실험하면서 해당 갑 바꿀 수 있으므로 함수로 계산해서 넣도록 함 )

def calc_interaction_value(star, sentiment, star_weight=1.0, sentiment_weight=1.0):
    # ponytail: offline interaction 설계는 실험 대상으로, 기본은 단순 가중합(추후 클리핑/정규화 정책 확장 가능)
    return star_weight * star + sentiment_weight * sentiment


# Unit 5
# interaction_value는 review_df[star] + review_df[sentiment]로 계산해 triples에 사용
review_df["interaction_value"] = calc_interaction_value(
    pd.to_numeric(review_df["star"], errors="coerce").fillna(0.0),
    pd.to_numeric(review_df["sentiment"], errors="coerce").fillna(0.0),
)

triples = list(
    zip(
        review_df["group_id"].astype(str),
        review_df["recipe_id"].astype(str),
        review_df["interaction_value"].astype(float),
    )
)

interactions, _ = dataset.build_interactions(triples)
num_users, num_items = interactions.shape
density = interactions.nnz / (num_users * num_items) if num_users and num_items else 0.0

print({"shape": interactions.shape, "nnz": interactions.nnz, "density": density})

{'shape': (821, 563), 'nnz': 990, 'density': 0.0021418233190472996}


### Unit 6 - train/test 분할

In [17]:
# Unit 6
train, test = random_train_test_split(
    interactions,
    test_percentage=TEST_RATIO,
    random_state=np.random.RandomState(SEED),
)

print({"train_nnz": train.nnz, "test_nnz": test.nnz})

{'train_nnz': 792, 'test_nnz': 198}


### Unit 7 - 학습

In [19]:
# Unit 7
model = LightFM(loss="warp", random_state=SEED)
precisions = []

for epoch in range(EPOCHS):
    model.fit_partial(train, epochs=1, num_threads=NUM_THREADS)
    precision = precision_at_k(model, train, k=5).mean()
    precisions.append(float(precision))
    print(f"Epoch {epoch+1}/{EPOCHS}, Precision@5: {precision:.4f}")

: 

### Unit 8 - Precision/Recall 평가

In [ ]:
# Unit 8
metrics = {
    "precision@5": float(precision_at_k(model, test, k=5).mean()),
    "precision@10": float(precision_at_k(model, test, k=10).mean()),
    "recall@5": float(recall_at_k(model, test, k=5).mean()),
    "recall@10": float(recall_at_k(model, test, k=10).mean()),
}
print(metrics)

### Unit 9 - 실험 리포트 기록

In [ ]:
# Unit 10
experiment_report = {
    "data_path": str(DATA_PATH),
    "target_mode": TARGET_MODE,
    "seed": SEED,
    "test_ratio": TEST_RATIO,
    "epochs": EPOCHS,
    "loss": "warp",
    "matrix": {
        "num_users": int(interactions.shape[0]),
        "num_items": int(interactions.shape[1]),
        "nnz": int(interactions.nnz),
    },
    "metrics": metrics,
    "decision": go_no_go,
}

print(experiment_report)